# toxicity-classifier-finetuned — Colab Training

Fine-tunes DistilBERT and RoBERTa on Jigsaw Toxic Comments (T4/A100 runtime).
Saves checkpoints to Google Drive.

**Estimated cost:** ~$1–2 on T4 (30–45 min total)

**Before running:**
1. Upload `train.csv` from the Jigsaw Kaggle dataset to your Drive at `My Drive/jigsaw/train.csv`
2. Set `Runtime > Change runtime type` to GPU (T4 or A100)

In [ ]:
# Install dependencies
!pip install torch transformers datasets scikit-learn typer pydantic-settings pandas accelerate -q

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

JIGSAW_DATA_DIR = Path('/content/drive/MyDrive/jigsaw')
OUTPUT_DIR = Path('/content/drive/MyDrive/toxicity-checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['JIGSAW_DATA_DIR'] = str(JIGSAW_DATA_DIR)
os.environ['MODEL_OUTPUT_DIR'] = str(OUTPUT_DIR)

print(f'Data dir: {JIGSAW_DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'train.csv exists: {(JIGSAW_DATA_DIR / "train.csv").exists()}')

In [ ]:
# Clone or copy the toxicity_classifier package
# Option A: clone from your GitHub repo (replace with your repo URL after publishing)
# !git clone https://github.com/YOUR_USERNAME/toxicity-classifier-finetuned.git
# %cd toxicity-classifier-finetuned

# Option B: upload the toxicity_classifier/ directory to Colab files panel, then:
import sys

sys.path.insert(0, '/content')  # adjust if needed

In [ ]:
# Train DistilBERT — full dataset, 4 epochs
from toxicity_classifier.train_distilbert import train as train_distilbert

checkpoint_distilbert = train_distilbert(
    data_dir=JIGSAW_DATA_DIR,
    output_dir=OUTPUT_DIR,
    epochs=4,
    batch_size=32,
)
print(f'DistilBERT checkpoint: {checkpoint_distilbert}')

In [ ]:
# Train RoBERTa — full dataset, 4 epochs
from toxicity_classifier.train_roberta import train as train_roberta

checkpoint_roberta = train_roberta(
    data_dir=JIGSAW_DATA_DIR,
    output_dir=OUTPUT_DIR,
    epochs=4,
    batch_size=16,
)
print(f'RoBERTa checkpoint: {checkpoint_roberta}')

In [ ]:
# Evaluate both models
from toxicity_classifier.evaluate import evaluate

result_distilbert = evaluate(
    checkpoint_dir=OUTPUT_DIR / 'distilbert-best',
    model_key='distilbert',
    data_dir=JIGSAW_DATA_DIR,
    output_dir=OUTPUT_DIR / 'eval_results',
)

result_roberta = evaluate(
    checkpoint_dir=OUTPUT_DIR / 'roberta-best',
    model_key='roberta',
    data_dir=JIGSAW_DATA_DIR,
    output_dir=OUTPUT_DIR / 'eval_results',
)

In [ ]:
# Optional: zip and download checkpoints to local machine
from google.colab import files  # noqa: E402
import shutil

shutil.make_archive('/content/toxicity-checkpoints', 'zip', str(OUTPUT_DIR))
files.download('/content/toxicity-checkpoints.zip')